In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from eofs.multivariate.standard import MultivariateEof
from eofs.xarray import Eof

from utils_mitgcm import open_mitgcm_ds_from_config
from utils_signal_processing import *

In [ ]:
lake = 'neuchatel'
model = f'{lake}_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('../../config.json', model)

In [ ]:
horizontal_resolution = ds.dxC.isel(XG=100, YC=40).values
ds['YG'] = np.arange(0, len(ds['YG'])) * horizontal_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * horizontal_resolution
ds['YC'] = np.arange(1, len(ds['YC'])+1) * horizontal_resolution - horizontal_resolution/2
ds['XC'] = np.arange(1, len(ds['XC'])+1) * horizontal_resolution - horizontal_resolution/2

In [ ]:
depths = range(len(ds.Z))
times = range(10*24, 15*24) #range(15*24, len(ds.time))
ds_select = ds.isel(Z=depths, Zl = depths, time=times)

In [ ]:
aligned_u = ds_select.UVEL.rename({'XG':'XC'})
aligned_u['XC'] = ds_select['XC']

aligned_v = ds_select.VVEL.rename({'YG':'YC'})
aligned_v['YC'] = ds_select['YC']

aligned_w = ds_select.WVEL.rename({'Zl':'Z'})
aligned_w['Z'] = ds_select['Z']

In [ ]:
folder_path = os.path.dirname(mitgcm_config['datapath'])
output_folder = os.path.join(folder_path, "eof")
os.makedirs(output_folder, exist_ok=True)

# Rotary EOF with complex variable

In [ ]:
def rotary_eof_xarray(U, V, dA, n_modes=None):
    """
    Perform complex (rotary) EOF analysis on xarray Dataset with UVEL, VVEL.

    Parameters
    ----------
    ds : xarray.Dataset
        Must contain variables 'UVEL' and 'VVEL' with dims (time, z, y, x)
    dx, dy : float
        Horizontal grid spacing in meters (assumed uniform)
    rho : float or array-like, optional
        Density [kg/m³], either scalar or 1D array over z.
        Default: 1025 kg/m³ constant.
    n_modes : int, optional
        Number of EOF modes to retain. If None, keep all.

    Returns
    -------
    result : xarray.Dataset
        Contains complex EOFs (real=u pattern, imag=v pattern),
        principal components (PCs), modal KE, total KE, and explained variance.
    """

    # === 1. Align velocities ===
    dz = U.drF
    nt, nz, ny, nx = U.shape

    # === 2. Prepare weights ===
    rho_arr = xr.DataArray(
        np.full(dz.size, 1025.0),
        coords={"Z": dz["Z"]},
        dims=("Z",),
    )

    # Volume weight: sqrt(rho * dz * dA)
    w_z = np.sqrt(rho_arr * dz * dA)

    # complex weighted velocities
    Uw = (U * w_z).values.reshape(nt, nz*ny*nx)
    Vw = (V * w_z).values.reshape(nt, nz*ny*nx)
    Z = (Uw + 1j*Vw).T  # shape (nstate, nt)

    # === 3. Complex SVD ===
    Umat, S, VT = np.linalg.svd(Z, full_matrices=False)
    n_total = S.size
    if n_modes is None or n_modes > n_total:
        n_modes = n_total

    # truncate
    phi = Umat[:, :n_modes]
    PCs = (phi.conj().T @ Z).T  # (nt, n_modes)
    modal_ke_ts = 0.5 * np.abs(PCs) ** 2
    total_ke_ts = 0.5 * np.sum(np.abs(Z) ** 2, axis=0)

    # === 4. Map EOFs back to physical units ===
    w_vec = np.repeat(w_z.values, ny * nx)
    phi_phys = phi / w_vec[:, None]
    EOFs_phys = phi_phys.reshape(nz, ny, nx, n_modes).transpose(3, 0, 1, 2)

    # === 5. Compute explained variance fraction ===
    lam = 0.5 * S**2  # KE per mode
    frac = lam / lam.sum()

    # === 6. Package into xarray ===
    coords = {
        "mode": np.arange(1, n_modes + 1),
        "Z": U.Z,
        "YC": U.YC,
        "XC": U.XC,
        "time": U.time,
    }

    ds_out = xr.Dataset(
        {
            "EOF": (("mode", "Z", "YC", "XC"), EOFs_phys),
            "PC": (("time", "mode"), PCs),
            "modal_KE": (("time", "mode"), modal_ke_ts),
            "KE_total": ("time", total_ke_ts),
            "variance_fraction": ("mode", frac[:n_modes]),
        },
        coords=coords,
        attrs={
            "description": "Complex (rotary) EOF decomposition of UVEL + iVVEL",
            "weighting": "sqrt(rho * dz * dx * dy)",
            "units": {
                "EOF": "m/s (complex: real=u, imag=v)",
                "PC": "sqrt(J/kg)",
                "modal_KE": "J/kg",
            },
        },
    )

    return ds_out


In [ ]:
rotary = rotary_eof_xarray(aligned_u, aligned_v, 100*100, n_modes=5)

In [ ]:
# select first mode
mode = 0
EOF = rotary.EOF.isel(mode=mode)

# U and V components
U_pattern = EOF.real
V_pattern = EOF.imag

# Compute horizontal amplitude
amp = np.sqrt(U_pattern**2 + V_pattern**2)

# Plot quiver for a single depth slice
z_slice = 0  # e.g., 5th vertical level
plt.figure(figsize=(15,6))
subsetting_factor=5
amp.isel(Z=z_slice).plot(add_colorbar=False)
plt.quiver(rotary.XC[::subsetting_factor], rotary.YC[::subsetting_factor],
           U_pattern[z_slice,:,:][::subsetting_factor,::subsetting_factor],
           V_pattern[z_slice,:,:][::subsetting_factor,::subsetting_factor],
           scale=1e-5)
plt.gca().invert_yaxis()
plt.title(f'Rotary EOF mode {mode+1}, depth level {z_slice}')
plt.xlabel('X')
plt.ylabel('Y')
plt.colorbar(label='Amplitude')
plt.show()

PC = rotary.PC.isel(mode=mode)
plt.figure(figsize=(10,4))
plt.plot(rotary.time, PC, label='Amplitude')
plt.ylabel('Amplitude (sqrt(KE))')
plt.xlabel('Time')
plt.title(f'PC amplitude, mode {mode+1}')
plt.grid()
plt.show()


In [ ]:
def project_rotary_mode(
    U,
    V,
    dA,
    eof_u,
    eof_v,
    rho=1025.0,
    remove_mean=True,
    normalize=True,
):
    """
    Project MITgcm velocity fields onto a prescribed Kelvin-wave mode
    and compute its time-dependent amplitude and kinetic energy.

    Parameters
    ----------


    eof_u, eof_v : xarray.DataArray
        Kelvin mode velocity structure (z, y, x) on tracer grid (XC, YC)

    rho : float or array-like
        Density [kg/m^3]; can be scalar or function of depth

    remove_mean : bool
        Remove time mean before projection

    normalize : bool
        Normalize mode to unit energy norm

    Returns
    -------
    A : xarray.DataArray (time,) complex
        Kelvin wave amplitude

    KE : xarray.DataArray (time,)
        Kinetic energy of Kelvin mode
    """

    # --- 2. Align with EOF grid ---
    U, eof_u = xr.align(U, eof_u, join="exact")
    V, eof_v = xr.align(V, eof_v, join="exact")

    # --- 4. Build complex fields ---
    Z = U + 1j * V
    phi = eof_u + 1j * eof_v

    # --- 5. Build volume weights ---
    dz = U["drF"]

    rho_arr = xr.DataArray(
        np.full(dz.size, rho),
        coords={"Z": dz["Z"]},
        dims=("Z",),
    )

    # Volume weight: sqrt(rho * dz * dA)
    w = np.sqrt(rho_arr * dz * dA)

    # --- 6. Apply weights ---
    Zw = (Z * w).stack(space=("Z", "YC", "XC"))
    phiw = (phi * w).stack(space=("Z", "YC", "XC"))

    # Convert to numpy for fast linear algebra
    Zw_np = Zw.values
    phiw_np = phiw.values

    # --- 7. Normalize mode ---
    if normalize:
        norm = np.sqrt(np.vdot(phiw_np, phiw_np).real)
        if norm == 0:
            raise ValueError("Mode has zero norm.")
        phiw_np = phiw_np / norm

    # --- 8. Projection (complex amplitude) ---
    A = Zw_np @ phiw_np.conj()

    # --- 9. Kinetic energy ---
    KE = 0.5 * np.abs(A) ** 2

    # --- 10. Wrap outputs ---
    A_da = xr.DataArray(A, coords={"time": U["time"]}, dims=("time",), name="A_kelvin")
    KE_da = xr.DataArray(KE, coords={"time": U["time"]}, dims=("time",), name="KE_kelvin")

    return A_da, KE_da

In [ ]:
times = range(0, 31*24) #range(15*24, len(ds.time))
ds_select = ds.isel(Z=depths, Zl = depths, time=times)

In [ ]:
aligned_u = ds_select.UVEL.rename({'XG':'XC'})
aligned_u['XC'] = ds_select['XC']

aligned_v = ds_select.VVEL.rename({'YG':'YC'})
aligned_v['YC'] = ds_select['YC']

aligned_w = ds_select.WVEL.rename({'Zl':'Z'})
aligned_w['Z'] = ds_select['Z']

In [ ]:
pc_proj, ke_proj = project_rotary_mode(aligned_u, aligned_v, 100*100, U_pattern, V_pattern, normalize=True)

In [ ]:
PC = rotary.PC.isel(mode=mode)
plt.figure(figsize=(10,4))

plt.plot(rotary.time, PC, label='PC EOF')
plt.plot(pc_proj.time, pc_proj, label='Amplitude projection')
plt.ylabel('Amplitude (sqrt(KE))')
plt.xlabel('Time')
plt.legend()
plt.title(f'PC amplitude, mode {mode+1}')
plt.grid()
plt.show()

In [ ]:
pc_proj.shape[0]

In [ ]:
plt.plot(ke_proj.time, ke_proj, label='KE projection')

In [ ]:
(ke_proj/1e6).to_dataframe(name='kinetic_energy_[MJ]').reset_index().to_csv(os.path.join(output_folder, "ke_mode1.csv"))

In [ ]:
mode=0
PC = pc_proj

# amplitude
plt.figure(figsize=(10,4))
plt.plot(PC.time, np.abs(PC), label='Amplitude')
plt.ylabel('Amplitude (sqrt(KE))')
plt.xlabel('Time')
plt.title(f'PC amplitude, mode {mode+1}')
plt.grid()
plt.show()

# phase (rotation)
plt.figure(figsize=(10,4))
plt.plot(PC.time, np.angle(PC), label='Phase')
plt.ylabel('Phase (radians)')
plt.xlabel('Time')
plt.title(f'PC phase, mode {mode+1}')
plt.grid()
plt.show()


In [ ]:
for time_index in range(5*24, 15*24):
    time_sel = pc_proj.time.isel(time=time_index)

    z_index = int(z_slice) if "z_slice" in globals() else 0
    z_sel = U_pattern.Z.isel(Z=z_index)

    dA = float(horizontal_resolution) ** 2
    rho = 1025.0

    dz = aligned_u["drF"]
    rho_arr = xr.DataArray(np.full(dz.size, rho), coords={"Z": dz["Z"]}, dims=("Z",))
    w = np.sqrt(rho_arr * dz * dA)

    phi = (U_pattern + 1j * V_pattern)
    phiw = (phi * w).stack(space=("Z", "YC", "XC"))
    norm = np.sqrt(np.vdot(phiw.values, phiw.values).real)
    phi_n = phi / norm

    A_t = pc_proj.sel(time=time_sel)

    vel_hat = (A_t * phi_n).sel(Z=z_sel)
    u_hat = vel_hat.real
    v_hat = vel_hat.imag
    spd_hat = np.sqrt(u_hat ** 2 + v_hat ** 2)

    time_sel, z_sel, float(np.abs(A_t.values))
    plt.figure(figsize=(15, 6))

    vmax = float(spd_hat.max())
    spd_hat.plot(cmap="viridis", vmin=0, vmax=vmax, add_colorbar=True)

    plt.title(
        f"Reconstructed velocities from pc_proj (time={np.datetime_as_string(time_sel.values)}, Z={float(z_sel.values):.3g})")
    plt.xlabel("X [m]")
    plt.ylabel("Y [m]")

    sf = int(subsetting_factor) if "subsetting_factor" in globals() else 5
    plt.quiver(
        spd_hat["XC"][::sf],
        spd_hat["YC"][::sf],
        u_hat.values[::sf, ::sf],
        v_hat.values[::sf, ::sf],
        color="white",
        scale=None,
    )

    plt.gca().invert_yaxis()
    plt.savefig(os.path.join(output_folder, f"pc_proj_mode{mode+1}_time{time_index}.png"))
    plt.close('all')


In [ ]:
from numpy.fft import fft, fftfreq

pc = PC.values
nt = len(rotary.time)
dt = (rotary.time[1] - rotary.time[0]).values / np.timedelta64(1, 's')  # seconds as float

freq = fftfreq(nt, d=dt)
spectrum = np.abs(fft(pc))**2

cw_energy = spectrum[freq < 0].sum()
ccw_energy = spectrum[freq > 0].sum()

rotation = "CCW" if ccw_energy > cw_energy else "CW"
print(f"Mode {mode+1} is predominantly {rotation} rotating")


In [ ]:
plt.figure(figsize=(10,4))
for m in range(rotary.dims['mode']):
    plt.plot(rotary.time, rotary.modal_KE.isel(mode=m), label=f'Mode {m+1}')
plt.ylabel('Modal KE')
plt.xlabel('Time')
plt.title('Rotary EOF Modal KE')
plt.legend()
plt.grid()
plt.show()